In [34]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df_tf = dataset_limpo_medianag.copy()
base_total = len(df_tf)

candidate_cols = [
    'WebframeHaveWorkedWith',
    'MiscTechHaveWorkedWith'
]

tech_cols = [c for c in candidate_cols if c in df_tf.columns]

if not tech_cols:
    raise ValueError('Nenhuma coluna de web frameworks/technologies foi encontrada no dataset.')

counts_by_tech = {}

for _, row in df_tf[tech_cols].fillna('').iterrows():
    per_respondent = set()
    
    for col in tech_cols:
        raw_value = str(row[col])
        
        for item in raw_value.split(';'):
            tech = item.strip()
            if tech and tech != 'Sem dado':
                per_respondent.add(tech)
    
    for tech in per_respondent:
        counts_by_tech[tech] = counts_by_tech.get(tech, 0) + 1

all_counts = pd.Series(counts_by_tech).sort_values(ascending=False)
all_pct = (all_counts / base_total) * 100

validation_all_df = pd.DataFrame({
    'Technology': all_counts.index,
    'Count': all_counts.values,
    'Percent': all_pct.values
})

# Top 6
plot_df = validation_all_df.head(6).copy()

# Labels
label_map = {
    'Node.js': 'Node.js',
    'React': 'React',
    'Angular': 'Angular',
    'Vue.js': 'Vue.js',
    'jQuery': 'jQuery',
    'ASP.NET CORE': 'ASP.NET Core'
}

plot_df['Label'] = plot_df['Technology'].map(label_map).fillna(plot_df['Technology'])

colors = ['#1e88e5', '#14b8a6', '#fbbf24', '#fb7f3f', '#7c78d1', '#84c69b']

# === PIE BASE ===
fig = go.Figure()

fig.add_trace(go.Pie(
    labels=plot_df['Label'],
    values=plot_df['Percent'],
    marker=dict(colors=colors, line=dict(color='#ffffff', width=2)),
    textinfo='none',
    hovertemplate='%{label}<br>%{value:.2f}%<extra></extra>',
    sort=False
))

# === POSICIONAR ETIQUETAS COMO EN TU GRÁFICO ===
annotations = []

for i in range(len(plot_df)):
    angle = (sum(plot_df['Percent'][:i]) + plot_df['Percent'].iloc[i] / 2) / 100 * 360
    angle_rad = np.deg2rad(90 - angle)

    # punto en el borde
    x = 1.1 * np.cos(angle_rad)
    y = 1.1 * np.sin(angle_rad)

    # posición del texto
    x_text = 1.4 * np.cos(angle_rad)
    y_text = 1.4 * np.sin(angle_rad)

    annotations.append(dict(
        x=x_text,
        y=y_text,
        text=f"{plot_df['Label'].iloc[i]}",
        showarrow=True,
        arrowhead=0,
        ax=x,
        ay=y,
        arrowcolor=colors[i],
        font=dict(size=13, color=colors[i], family="Arial")
    ))

# Layout estilo "card"
fig.update_layout(
    title=dict(
        text="Popular Technologies & Frameworks<br><sup>Most commonly used in production</sup>",
        x=0.5
    ),
    showlegend=False,
    annotations=annotations,
    template="plotly_white",
    margin=dict(l=40, r=40, t=80, b=40),
    width=800,
    height=800
)

# Exportar HTML interactivo
fig.write_html(
    "./Validations/web_frameworks_technologies_plotly.html"
)

# Guardar CSV
validation_all_df.to_csv(
    './Validations/web_frameworks_technologies_all_validation.csv',
    index=False,
    encoding='utf-8-sig'
)

# Prints
print('Base total do dataset:', base_total)
print('Colunas usadas:', ', '.join(tech_cols))
print('Total de opções validadas:', len(validation_all_df))
print('\nTop 6 (% real):')
print(plot_df[['Label', 'Percent']].to_string(index=False))

Base total do dataset: 65437
Colunas usadas: WebframeHaveWorkedWith, MiscTechHaveWorkedWith
Total de opções validadas: 75

Top 6 (% real):
    Label   Percent
  Node.js 30.215322
    React 29.290768
.NET (5+) 17.638339
   jQuery 15.864114
    NumPy 14.850925
   Pandas 14.519309
